# Strands Agents with Bedrock AgentCore Code Interpreter — FSI Edition

This lab demonstrates how to use Amazon Bedrock AgentCore Code Interpreter to give your AI agent the ability to execute Python code dynamically — applied to financial services use cases.

## Overview

In this lab, you will:
- Use the default Code Interpreter to run financial calculations in a sandbox
- Analyze transaction data for fraud patterns
- Calculate portfolio risk metrics (VaR, sector concentration)
- Create a custom Code Interpreter with network access for live market data

## Why Code Interpreter for FSI?

Financial services require:
- **Dynamic calculations** — Risk models, stress tests, scenario analysis
- **Data analysis** — Fraud detection, anomaly identification
- **Secure execution** — Sandboxed environment for sensitive financial data
- **Audit trail** — Every calculation is traceable

## Prerequisites

Ensure you have AWS credentials configured and Nova Pro model access enabled.

In [ ]:
import os

#os.environ["AWS_ACCESS_KEY_ID"] = ""
#os.environ["AWS_SECRET_ACCESS_KEY"] = ""
#os.environ["AWS_SESSION_TOKEN"] = ""
#os.environ["AWS_REGION"] = ""

In [ ]:
#%pip install -q strands-agents strands-agents-tools rich bedrock-agentcore pandas

In [5]:
import boto3

region = boto3.session.Session().region_name

NOVA_PRO_MODEL_ID = "us.amazon.nova-pro-v1:0"
if region.startswith("eu"):
    NOVA_PRO_MODEL_ID = "eu.amazon.nova-pro-v1:0"
elif region.startswith("ap"):
    NOVA_PRO_MODEL_ID = "apac.amazon.nova-pro-v1:0"

print(f"Region: {region}")
print(f"Nova Pro Model ID: {NOVA_PRO_MODEL_ID}")

Region: ap-southeast-2
Nova Pro Model ID: apac.amazon.nova-pro-v1:0


## Part 1: Default Code Interpreter — Financial Calculations

The default Code Interpreter runs Python in a **sandboxed environment** with no network access. Perfect for secure financial calculations.

Let's test it with a portfolio risk calculation:

In [6]:
from strands import Agent
from strands.models import BedrockModel
from strands_tools.code_interpreter import AgentCoreCodeInterpreter

# Initialize the AgentCore Code Interpreter (default: sandboxed, no network)
agentcore_code_interpreter = AgentCoreCodeInterpreter()

# Create agent with default Code Interpreter (sandboxed)
risk_agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID, max_tokens=4096),
    system_prompt="""You are a quantitative analyst assistant. You write and execute Python code 
    to perform financial calculations. Keep responses concise.""",
    tools=[agentcore_code_interpreter.code_interpreter],
)

risk_agent("Calculate the future value of a $2,000,000 investment at 4.8% annual rate compounded monthly after 5 years.")

<thinking> To calculate the future value of the investment, I need to use the future value formula for compound interest, which is FV = PV * (1 + r/n)^(n*t), where PV is the present value, r is the annual interest rate, n is the number of times interest is compounded per year, and t is the number of years. In this case, PV = $2,000,000, r = 0.048, n = 12 (monthly compounding), and t = 5. I will use the executeCode action to perform the calculation. </thinking>

Tool #1: code_interpreter
The future value of the $2,000,000 investment at 4.8% annual rate compounded monthly after 5 years is approximately $2,541,281.44.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'The future value of the $2,000,000 investment at 4.8% annual rate compounded monthly after 5 years is approximately $2,541,281.44.'}], 'metadata': {'usage': {'inputTokens': 4139, 'outputTokens': 47, 'totalTokens': 4186}, 'metrics': {'latencyMs': 768, 'timeToFirstByteMs': 457}}}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={'code_interpreter': ToolMetrics(tool={'toolUseId': 'tooluse_7rOxtjLCB3vlp7Xqqr6fLE', 'name': 'code_interpreter', 'input': {'code_interpreter_input': {'action': {'type': 'executeCode', 'code': 'PV = 2000000\nr = 0.048\nn = 12\nt = 5\nFV = PV * (1 + r/n)**(n*t)\nprint(FV)'}}}}, call_count=1, success_count=1, error_count=0, total_time=1.5338490009307861)}, cycle_durations=[3.785313844680786, 0.7893199920654297], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='882de46b-11de-4b20-9e3c-cad18a0751a3', usage={'inputTokens': 3852, 'outputTokens

## Part 2: Fraud Detection on Transaction Data

Now let's give the agent our synthetic transaction dataset and ask it to identify fraud patterns.

The dataset (`data/transactions.csv`) contains 25 transactions with several suspicious patterns:
- **Velocity attack** — Multiple high-value transactions within seconds
- **Geo-anomaly** — Transactions in different countries within minutes
- **Escalating amounts** — Progressively larger transactions (testing limits)
- **Unusual timing** — High-value transactions at 3am

In [7]:
# Load the transaction data so we can pass it to the agent
import pandas as pd

transactions_df = pd.read_csv("../data/transactions.csv")
print(f"Loaded {len(transactions_df)} transactions")
transactions_df.head()

Loaded 25 transactions


,transaction_id,timestamp,customer_id,amount,currency,merchant,category,location_city,location_country,card_type,is_online
0,TXN-001,2026-05-28 08:15:23,CUST-4421,12.5,AUD,Morning Brew Cafe,Food & Drink,Sydney,AU,debit,False
1,TXN-002,2026-05-28 08:17:45,CUST-4421,3200.0,AUD,TechWorld Electronics,Electronics,Lagos,NG,debit,True
2,TXN-003,2026-05-28 08:18:12,CUST-4421,2800.0,AUD,GiftCards Express,Gift Cards,Lagos,NG,debit,True
3,TXN-004,2026-05-28 08:19:01,CUST-4421,1500.0,AUD,Crypto Exchange XYZ,Financial Services,Moscow,RU,debit,True
4,TXN-005,2026-05-28 12:30:00,CUST-4421,15.8,AUD,Lunch Spot,Food & Drink,Sydney,AU,debit,False


In [8]:
# Pass the data as context and ask the agent to analyze it
transaction_data = transactions_df.to_csv(index=False)

risk_agent(f"""Here is a CSV of transaction data:
{transaction_data}

Write Python code to detect fraud. Check for:
1. Velocity: >2 transactions within 5 minutes for same customer
2. Geo-anomaly: transactions in different countries within 30 minutes
3. Amount: single transaction > 000 or escalating pattern
4. Timing: transactions between midnight and 5am

Output a table with columns: transaction_id, customer_id, amount, flag_reason, risk_score (1-10).
Only show flagged transactions. Sort by risk_score descending.""")

Certainly! Below is the Python code to detect the specified fraud patterns and flag suspicious transactions with a risk score.

```python
import pandas as pd
from datetime import datetime, timedelta

# Transaction data
data = [
    ["TXN-001","2026-05-28 08:15:23","CUST-4421",12.5,"AUD","Morning Brew Cafe","Food & Drink","Sydney","AU","debit",False],
    ["TXN-002","2026-05-28 08:17:45","CUST-4421",3200.0,"AUD","TechWorld Electronics","Electronics","Lagos","NG","debit",True],
    ["TXN-003","2026-05-28 08:18:12","CUST-4421",2800.0,"AUD","GiftCards Express","Gift Cards","Lagos","NG","debit",True],
    ["TXN-004","2026-05-28 08:19:01","CUST-4421",1500.0,"AUD","Crypto Exchange XYZ","Financial Services","Moscow","RU","debit",True],
    ["TXN-005","2026-05-28 12:30:00","CUST-4421",15.8,"AUD","Lunch Spot","Food & Drink","Sydney","AU","debit",False],
    ["TXN-006","2026-05-28 09:00:00","CUST-7832",250.0,"AUD","Woolworths","Groceries","Melbourne","AU","credit",False],
    ["TXN-007","2026-05-

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'Certainly! Below is the Python code to detect the specified fraud patterns and flag suspicious transactions with a risk score.\n\n```python\nimport pandas as pd\nfrom datetime import datetime, timedelta\n\n# Transaction data\ndata = [\n    ["TXN-001","2026-05-28 08:15:23","CUST-4421",12.5,"AUD","Morning Brew Cafe","Food & Drink","Sydney","AU","debit",False],\n    ["TXN-002","2026-05-28 08:17:45","CUST-4421",3200.0,"AUD","TechWorld Electronics","Electronics","Lagos","NG","debit",True],\n    ["TXN-003","2026-05-28 08:18:12","CUST-4421",2800.0,"AUD","GiftCards Express","Gift Cards","Lagos","NG","debit",True],\n    ["TXN-004","2026-05-28 08:19:01","CUST-4421",1500.0,"AUD","Crypto Exchange XYZ","Financial Services","Moscow","RU","debit",True],\n    ["TXN-005","2026-05-28 12:30:00","CUST-4421",15.8,"AUD","Lunch Spot","Food & Drink","Sydney","AU","debit",False],\n    ["TXN-006","2026-05-28 09:00:00","CUST-

## Part 3: Portfolio Risk Analysis (VaR)

Let's analyze a portfolio using Value at Risk (VaR) — a standard risk metric in financial services.

We'll use the portfolio data from `data/portfolio.csv`.

In [ ]:
portfolio_df = pd.read_csv("../data/portfolio.csv")
print(f"Loaded {len(portfolio_df)} positions")
portfolio_df.head(10)

In [ ]:
portfolio_data = portfolio_df.to_csv(index=False)

risk_agent(f"""Analyze this portfolio for risk metrics. Calculate:
1. Total portfolio value (current prices × units) for each client
2. Sector concentration — what % is in each sector? Flag if any sector > 30%
3. Unrealized P&L per position (current vs purchase price)
4. Asset class allocation (Equity vs Fixed Income vs Cash vs Other)

Present results as a clear summary with any risk warnings.

Portfolio data:
{portfolio_data}""")

## Part 4: Custom Code Interpreter with Network Access

The default Code Interpreter is sandboxed (no internet). For use cases that need live data (e.g., fetching real stock prices), we create a **custom Code Interpreter with network access**.

### Step 1: Initialize AgentCore Clients

In [ ]:
from bedrock_agentcore.runtime import AgentCoreApp
from bedrock_agentcore.services.code_interpreter import CodeInterpreterService

# Initialize the Code Interpreter service
ci_service = CodeInterpreterService()

print("✅ AgentCore Code Interpreter service initialized")

### Step 2: Create Custom Code Interpreter with Network Access

In [ ]:
# Create a custom code interpreter with public network access
custom_ci = ci_service.create_code_interpreter(
    name="fsi-risk-analyzer",
    network_access="PUBLIC"
)

print(f"✅ Custom Code Interpreter created: {custom_ci.code_interpreter_id}")
print(f"   Network access: PUBLIC (can fetch live market data)")

### Step 3: Create a Session and Test Live Data Access

In [ ]:
# Create a session in the custom code interpreter
session = custom_ci.create_session()

print(f"✅ Session created: {session.session_id}")

# Test: install yfinance and fetch a real stock price
result = session.execute_code("""
import subprocess
subprocess.run(['pip', 'install', '-q', 'yfinance'], capture_output=True)

import yfinance as yf
cba = yf.Ticker('CBA.AX')
info = cba.info
print(f"CBA.AX Live Price: ${info.get('currentPrice', 'N/A')} AUD")
print(f"Market Cap: ${info.get('marketCap', 0)/1e9:.1f}B AUD")
print(f"P/E Ratio: {info.get('trailingPE', 'N/A')}")
""")

print(result.output)

### Step 4: Use Custom Code Interpreter with Strands Agent

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel

@tool
def execute_python(code: str) -> str:
    """Execute Python code in a secure sandbox with internet access.
    Can install packages with pip and fetch live data.
    
    Args:
        code: Python code to execute
    """
    result = session.execute_code(code)
    return result.output if result.output else "Code executed successfully (no output)"

# Create agent with custom code interpreter
live_agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
    system_prompt="""You are a quantitative analyst with access to live market data.
    You can execute Python code to fetch real-time prices, calculate risk metrics,
    and generate analysis. Use yfinance for market data.""",
    tools=[execute_python],
)

live_agent("Fetch the current prices of the big 4 Australian banks (CBA, WBC, NAB, ANZ) and compare their P/E ratios.")

## Examining the Agent Loop

In [ ]:
from rich.table import Table
import rich
import json

console = rich.get_console()

console.print("Agent Loop Detail")
console.rule()
console.print(f"Number of Loops: {live_agent.event_loop_metrics.cycle_count}")

table = Table(title="Agent Messages", show_lines=True)
table.add_column("Role", style="green")
table.add_column("Text", style="magenta", max_width=60)
table.add_column("Tool Name", style="cyan")
table.add_column("Tool Input", style="cyan", max_width=40)
table.add_column("Tool Result", style="cyan", max_width=40)

for message in live_agent.messages:
    text = [content["text"] for content in message["content"] if "text" in content]
    tool_name = [content["toolUse"]["name"] for content in message["content"] if "toolUse" in content]
    tool_input = [content["toolUse"]["input"] for content in message["content"] if "toolUse" in content]
    tool_result = [content["toolResult"]["content"][0] for content in message["content"] if "toolResult" in content]
    table.add_row(
        message["role"], (text[-1][:200] + "...") if text and len(text[-1]) > 200 else (text[-1] if text else ""),
        tool_name[-1] if tool_name else "",
        (json.dumps(tool_input[-1])[:150] + "...") if tool_input else "",
        (json.dumps(tool_result[-1])[:150] + "...") if tool_result else ""
    )

console.print(table)

## Resource Cleanup (Optional)

Clean up the custom Code Interpreter to avoid charges:

In [ ]:
# Uncomment to clean up
# session.close()
# custom_ci.delete()
# print("✅ Resources cleaned up")

## Summary

In this lab, you:

- ✅ Used the default Code Interpreter for secure financial calculations
- ✅ Analyzed transaction data for fraud patterns (velocity, geo-anomaly, timing)
- ✅ Calculated portfolio risk metrics (sector concentration, P&L, allocation)
- ✅ Created a custom Code Interpreter with network access for live market data
- ✅ Fetched real-time stock prices and compared bank P/E ratios

### FSI Takeaways

| Capability | FSI Application |
|-----------|----------------|
| Sandboxed execution | Secure risk calculations on sensitive data |
| Dynamic code generation | Ad-hoc analysis without pre-built reports |
| Network-enabled interpreter | Live market data, API integrations |
| Audit trail (agent loop) | Compliance — every calculation is traceable |

### Next: Lab 02 — Browser Automation
We'll use AgentCore Browser to monitor regulatory websites (APRA, ASX) and extract live financial data.